In [97]:
import csv
from neo4j import GraphDatabase
import pandas as pd
from IPython.display import display
from pyprojroot import here
from IPython.display import display
pd.set_option('display.max_colwidth', None)

In [98]:
NEO4J_URI = "neo4j://localhost:7687"
USER = "neo4j"
PASSWORD = "<your password>"

driver = GraphDatabase.driver(NEO4J_URI, auth=(USER, PASSWORD))
#session = driver.session(database="neo4j")

In [99]:
def run_df(query, **params) -> pd.DataFrame:
    """Run query, return DataFrame (opens its own session)."""
    with driver.session(database="neo4j") as s:
        res = s.run(query, **params)
        return pd.DataFrame([r.values() for r in res], columns=res.keys())

def run_tx(query, **params):
    """Run write query (opens its own session)."""
    with driver.session(database="neo4j") as s:
        s.execute_write(lambda tx: tx.run(query, **params))

In [100]:
def run_tx(query, **params):
    with driver.session(database="neo4j") as s:
        s.execute_write(lambda tx: tx.run(query, **params))


In [101]:

#Delete all data

run_tx("MATCH (n) DETACH DELETE n")

# drop constraints / indexes if they exist
for q in [
    "DROP CONSTRAINT stations_uic_unique IF EXISTS",
    "DROP INDEX stations_name_idx IF EXISTS",
    "DROP CONSTRAINT line_code_norm_unique IF EXISTS",
    "DROP CONSTRAINT line_code_unique IF EXISTS"
]:
    run_tx(q)

display(run_df("SHOW CONSTRAINTS"))
display(run_df("SHOW INDEXES"))


,id,name,type,entityType,labelsOrTypes,properties,ownedIndex,propertyType
0,4,station_unique,UNIQUENESS,NODE,[Station],"[name, source]",station_unique,None


,id,name,state,populationPercent,type,entityType,labelsOrTypes,properties,indexProvider,owningConstraint,lastRead,readCount
0,1,index_343aff4e,ONLINE,100.0,LOOKUP,NODE,None,None,token-lookup-1.0,None,2025-12-06T12:46:36.241000000+00:00,82294
1,2,index_f7700477,ONLINE,100.0,LOOKUP,RELATIONSHIP,None,None,token-lookup-1.0,None,2025-12-03T19:37:09.410000000+00:00,254
2,3,station_unique,ONLINE,100.0,RANGE,NODE,[Station],"[name, source]",range-1.0,station_unique,2025-11-24T16:37:14.190000000+00:00,43024


In [102]:
# Create indexes and constants due to performance issues. If this isn't here neo4j is exteremely slow
run_tx("""
CREATE CONSTRAINT stations_uic_unique IF NOT EXISTS
FOR (s:Stations) REQUIRE s.uic IS UNIQUE
""")

run_tx("""
CREATE INDEX stations_name_idx IF NOT EXISTS
FOR (s:Stations) ON (s.name)
""")

run_tx("""
CREATE CONSTRAINT line_code_unique IF NOT EXISTS
FOR (l:Line) REQUIRE l.line_code IS UNIQUE
""")

run_tx("""
CREATE CONSTRAINT line_code_norm_unique IF NOT EXISTS
FOR (l:Line) REQUIRE l.line_code_norm IS UNIQUE
""")

display(run_df("SHOW CONSTRAINTS"))
display(run_df("SHOW INDEXES"))



,id,name,type,entityType,labelsOrTypes,properties,ownedIndex,propertyType
0,11,line_code_norm_unique,UNIQUENESS,NODE,[Line],[line_code_norm],line_code_norm_unique,None
1,9,line_code_unique,UNIQUENESS,NODE,[Line],[line_code],line_code_unique,None
2,4,station_unique,UNIQUENESS,NODE,[Station],"[name, source]",station_unique,None
3,6,stations_uic_unique,UNIQUENESS,NODE,[Stations],[uic],stations_uic_unique,None


,id,name,state,populationPercent,type,entityType,labelsOrTypes,properties,indexProvider,owningConstraint,lastRead,readCount
0,1,index_343aff4e,ONLINE,100.0,LOOKUP,NODE,None,None,token-lookup-1.0,None,2025-12-06T12:46:36.241000000+00:00,82294.0
1,2,index_f7700477,ONLINE,100.0,LOOKUP,RELATIONSHIP,None,None,token-lookup-1.0,None,2025-12-03T19:37:09.410000000+00:00,254.0
2,10,line_code_norm_unique,ONLINE,100.0,RANGE,NODE,[Line],[line_code_norm],range-1.0,line_code_norm_unique,None,NaN
3,8,line_code_unique,ONLINE,100.0,RANGE,NODE,[Line],[line_code],range-1.0,line_code_unique,None,NaN
4,3,station_unique,ONLINE,100.0,RANGE,NODE,[Station],"[name, source]",range-1.0,station_unique,2025-11-24T16:37:14.190000000+00:00,43024.0
5,7,stations_name_idx,ONLINE,100.0,RANGE,NODE,[Stations],[name],range-1.0,None,None,0.0
6,5,stations_uic_unique,ONLINE,100.0,RANGE,NODE,[Stations],[uic],range-1.0,stations_uic_unique,None,0.0


In [103]:
# load lines
lines_csv = here() / "data" / "processed" / "ines_per_region_sncf_national-p.csv"
print("Lines CSV exists?", lines_csv.exists(), lines_csv)

with lines_csv.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

cypher_lines = """
UNWIND $rows AS row
WITH row
WHERE row.line_code IS NOT NULL AND row.line_code <> ''
MERGE (l:Line { line_code: toInteger(row.line_code) })
SET l.line_name      = coalesce(row.line_name, row.name),
    l.region_name    = row.region_name,
    l.line_code_norm = toString(l.line_code),

    // raw PK strings kept for debugging
    l.PKD            = row.PKD,
    l.PKF            = row.PKF,

    // PKD: start km in meters
    l.pkd_m =
      CASE
        WHEN row.PKD IS NULL OR row.PKD = '' THEN NULL
        WHEN row.PKD =~ '^[0-9]+\\+[0-9]+$'
          THEN toInteger(split(row.PKD,'+')[0])*1000 + toInteger(split(row.PKD,'+')[1])
        WHEN row.PKD =~ '^[0-9]+-[0-9]+$'
          THEN toInteger(split(row.PKD,'-')[0])*1000 + toInteger(split(row.PKD,'-')[1])
        WHEN row.PKD =~ '^[0-9]+\\.[0-9]+$'
          THEN toInteger(toFloat(row.PKD)*1000)
        ELSE NULL    // D+000, H+566, etc.
      END,

    // PKF: end km in meters
    l.pkf_m =
      CASE
        WHEN row.PKF IS NULL OR row.PKF = '' THEN NULL
        WHEN row.PKF =~ '^[0-9]+\\+[0-9]+$'
          THEN toInteger(split(row.PKF,'+')[0])*1000 + toInteger(split(row.PKF,'+')[1])
        WHEN row.PKF =~ '^[0-9]+-[0-9]+$'
          THEN toInteger(split(row.PKF,'-')[0])*1000 + toInteger(split(row.PKF,'-')[1])
        WHEN row.PKF =~ '^[0-9]+\\.[0-9]+$'
          THEN toInteger(toFloat(row.PKF)*1000)
        ELSE NULL    // D+841, H+566, etc.
      END
"""

run_tx(cypher_lines, rows=rows)
display(run_df("MATCH (l:Line) RETURN count(l) AS lines"))
display(run_df("MATCH (l:Line) RETURN count(*) AS lines, count(l.pkd_m) AS with_pkd_m, count(l.pkf_m) AS with_pkf_m"))


Lines CSV exists? True data/processed/ines_per_region_sncf_national-p.csv


,lines
0,993


,lines,with_pkd_m,with_pkf_m
0,993,991,991


In [104]:
# load stations
stations_csv = here() / "data" / "processed" / "sncf_station_list_national-p.csv"
print("Stations CSV exists?", stations_csv.exists(), stations_csv)

with stations_csv.open(newline="", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

cypher_stations = """
UNWIND $rows AS row
WITH
  row,
  coalesce(row.station, row.name) AS station_name,
  CASE
    WHEN row.line_code IS NOT NULL AND row.line_code <> '' THEN toInteger(row.line_code)
    WHEN row.line      IS NOT NULL AND row.line      <> '' THEN toInteger(row.line)
    ELSE NULL
  END AS lc
WHERE row.uic IS NOT NULL AND row.uic <> '' AND station_name IS NOT NULL

MERGE (s:Stations { uic: toInteger(row.uic) })
SET s.name       = station_name,

    // WGS84 degrees from CSV (X_WGS84 = lon, Y_WGS84 = lat)
    s.lon        = CASE
                     WHEN row.X_WGS84 IS NULL OR row.X_WGS84 = '' THEN NULL
                     ELSE toFloat(row.X_WGS84)
                   END,
    s.lat        = CASE
                     WHEN row.Y_WGS84 IS NULL OR row.Y_WGS84 = '' THEN NULL
                     ELSE toFloat(row.Y_WGS84)
                   END,

    s.PK         = coalesce(row.PK, row.pk),
    s.line_code  = lc,
    s.line_code_norm = CASE WHEN lc IS NULL THEN NULL ELSE toString(lc) END,

    // Neo4j point() from lon/lat
    s.location   = CASE
                     WHEN row.X_WGS84 IS NOT NULL AND row.X_WGS84 <> ''
                      AND row.Y_WGS84 IS NOT NULL AND row.Y_WGS84 <> ''
                     THEN point({
                       longitude: toFloat(row.X_WGS84),
                       latitude:  toFloat(row.Y_WGS84)
                     })
                     ELSE NULL
                   END
"""

run_tx(cypher_stations, rows=rows)

display(run_df("""
MATCH (s:Stations)
RETURN count(*) AS stations,
       count(s.location) AS with_location
"""))


Stations CSV exists? True data/processed/sncf_station_list_national-p.csv


,stations,with_location
0,6062,6062


In [105]:
# Robust PK parsing for stations: some stations have a different prefix adusting here for that pattern
# 123+456 -> 123456 m; 000-031 -> 31 m; 12.345 -> 12345 m; D+936 -> NULL
run_tx("""
MATCH (s:Stations)
WITH s, s.PK AS raw
WITH s,
     CASE
       WHEN raw IS NULL OR raw = '' THEN NULL
       WHEN raw =~ '^[0-9]+\\+[0-9]+$'
         THEN toInteger(split(raw,'+')[0])*1000 + toInteger(split(raw,'+')[1])
       WHEN raw =~ '^[0-9]+-[0-9]+$'
         THEN toInteger(split(raw,'-')[0])*1000 + toInteger(split(raw,'-')[1])
       WHEN raw =~ '^[0-9]+\\.[0-9]+$'
         THEN toInteger(toFloat(raw)*1000)
       ELSE NULL   // D+936, F+489, H+566
     END AS pk_val
SET s.pk_m = pk_val
""")

display(run_df("""
MATCH (s:Stations)
RETURN count(*) AS stations, count(s.pk_m) AS with_pk_m
"""))

display(run_df("""
MATCH (s:Stations)
WHERE s.pk_m IS NULL
RETURN s.name AS station, s.PK AS raw_PK, s.line_code AS line_code, s.uic AS uic
ORDER BY station
"""))



,stations,with_pk_m
0,6062,6060


,station,raw_PK,line_code,uic
0,Musée-d'Orsay,D+936,984000,87547307
1,St-Michel-Notre-Dame,F+489,984000,87547315


In [106]:
# Establish relatioship between stations and lines using ON_LINE variable
run_tx("""
MATCH (s:Stations)
WHERE s.line_code_norm IS NOT NULL
MATCH (l:Line { line_code_norm: s.line_code_norm })
MERGE (s)-[:ON_LINE]->(l)
""")

display(run_df("MATCH (:Stations)-[r:ON_LINE]->(:Line) RETURN count(r) AS on_line_edges"))


,on_line_edges
0,5997


87313759

In [107]:

# Build TRACK edges by consecutive pk_m on each line, both directions
# This will group stations by line, sort stations along each line by pk_m. Then pairs each station with its next neighbor
# Computes distnce between neighbor from pk_m then creates bidiretional Track relationship.

run_tx("""
MATCH (l:Line)<-[:ON_LINE]-(s:Stations)
WHERE s.pk_m IS NOT NULL
WITH l, s
ORDER BY l.line_code_norm, s.pk_m
WITH l, collect(s) AS st
UNWIND range(0, size(st)-2) AS i
WITH l, st[i] AS a, st[i+1] AS b
WITH l, a, b, abs(b.pk_m - a.pk_m) AS d
MERGE (a)-[r1:TRACK { line_code: l.line_code }]->(b)
  ON CREATE SET r1.delta_m = d, r1.weight_m = d
MERGE (b)-[r2:TRACK { line_code: l.line_code }]->(a)
  ON CREATE SET r2.delta_m = d, r2.weight_m = d
""")

display(run_df("MATCH ()-[r:TRACK]-() RETURN count(r) AS track_edges"))

display(run_df("""
MATCH (s:Stations)-[r:TRACK]->(t:Stations)
RETURN s.name AS from, t.name AS to, r.line_code AS line, r.delta_m AS delta_m
LIMIT 10
"""))



,track_edges
0,21724


,from,to,line,delta_m
0,Cuzorn,St-Front-sur-Lémance,631000,5269
1,Cuzorn,Trentels-Ladignac,631000,12715
2,Gamarde-les-Bains,Hinx-sur-Adour,654000,4322
3,Gamarde-les-Bains,Montfort-en-Chalosse,654000,3220
4,Vars,St-Amant-de-Boixe,570000,5186
5,Vars,Angoulême,570000,13556
6,Monte-Carlo-Country-Club,Cap-d'Ail,930000,6552
7,Monte-Carlo-Country-Club,Cap-Martin-Roquebrune,930000,1007
8,La Boussac,Plerguer,415000,17184
9,La Boussac,Pleine-Fougères,415000,7480


In [108]:
# Create cross-region TRACK edges using geographic distance connects stations within 3km
run_tx("""
MATCH (a:Stations)
WHERE a.location IS NOT NULL

MATCH (b:Stations)
WHERE b.location IS NOT NULL
  AND id(b) > id(a)
  AND point.distance(a.location, b.location) < 3000 
  AND NOT (a)-[:TRACK]-(b)

WITH a, b, point.distance(a.location, b.location) AS d

MERGE (a)-[:TRACK {
  type: 'INTER_REGION',
  geo_dist_m: d
}]->(b)

MERGE (b)-[:TRACK {
  type: 'INTER_REGION',
  geo_dist_m: d
}]->(a);
""")

display(run_df("""
MATCH ()-[r:TRACK]-()
WHERE r.type = 'INTER_REGION'
RETURN count(r) AS inter_region_track_edges
"""))


Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description='The query used a deprecated function: `id`.', position=<SummaryInputPosition line=7, column=15, offset=112>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'column': 15, 'offset': 112, 'line': 7}}> for query: "\nMATCH (a:Stations)\nWHERE a.location IS NOT NULL\n\nMATCH (b:Stations)\nWHERE b.location IS NOT NULL\n  AND id(b) > id(a)\n  AND point.distance(a.location, b.location) < 3000 \n  AND NOT (a)-[:TRACK]-(b)\n\nWITH a, b, point.distance(a.location, b.location) AS d\n\nMERGE (a)-[:TRACK {\n  type: 'INTER_REGION',\n  geo_dist_m: d\n}]->(b)\n\nMERGE (b)-[:TRACK {\n  type: 'INTER_REGION',\n  geo_dist_m: d\n

,inter_region_track_edges
0,7004


In [109]:
# Degree distribution after adding INTER_REGION links
display(run_df("""
MATCH (s:Stations)
WITH s, size([(s)-[:TRACK]-() | 1]) AS degree
RETURN degree, count(*) AS stations
ORDER BY degree;
"""))

# Cross-region
display(run_df("""
MATCH (p:Stations {name:'St-Germain-les-Belles'}),
      (n:Stations {name:'Drulingen'})
MATCH path = shortestPath((p)-[:TRACK*..5]->(n))
RETURN length(path) AS hops,
       [x IN nodes(path) | x.name][0..25] AS first_25_stations;
"""))

# Paris -> Nice
display(run_df("""
MATCH (p:Stations {name:'Paris-Gare-de-Lyon'}),
      (n:Stations {name:'Nice-Ville'})
MATCH path = shortestPath((p)-[:TRACK*..4000]->(n))
RETURN length(path) AS hops,
       [x IN nodes(path) | x.name][0..25] AS first_25_stations;
"""))


,degree,stations
0,0,66
1,2,659
2,4,4371
3,6,448
4,8,173
5,10,74
6,12,63
7,14,35
8,16,34
9,18,24


,hops,first_25_stations


,hops,first_25_stations
0,194,"[Paris-Gare-de-Lyon, Paris-Bercy-Charolais, Ivry-sur-Seine, Maisons-Alfort-Alfortville, Le Vert-de-Maisons, Créteil-Pompadour, Villeneuve-Triage, Villeneuve-St-Georges, Montgeron-Crosne, Yerres, Brunoy, Boussy-St-Antoine, Combs-la-Ville-Quincy, Lieusaint-Moissy, Savigny-le-Temple-Nandy, Cesson, Le Mée, Bois-le-Roi, Fontainebleau-Avon, Thomery, St-Mammès, Moret-Veneux-les-Sablons, Montigny-sur-Loing, Bourron-Marlotte-Grez, Nemours-St-Pierre]"


In [110]:
#quick sanity check
display(run_df("MATCH (s:Stations) RETURN count(s) AS stations, count(s.pk_m) AS with_pk_m"))
display(run_df("MATCH (:Stations)-[:ON_LINE]->(:Line) RETURN count(*) AS on_line_edges"))
display(run_df("MATCH ()-[:TRACK]->() RETURN count(*) AS track_edges"))

# confirm Paris-Nice path is findable via existing TRACK edges
display(run_df("""
MATCH (p:Stations {name:'Paris-Gare-de-Lyon'}), (n:Stations {name:'Nice-Ville'})
MATCH path = shortestPath((p)-[:TRACK*..500]->(n))
RETURN length(path) AS hops
"""))


,stations,with_pk_m
0,6062,6060


,on_line_edges
0,5997


,track_edges
0,14364


,hops
0,194


In [111]:
display(run_df("""
MATCH (s:Stations)
RETURN count(*) AS stations, count(s.pk_m) AS with_pk_m
"""))

display(run_df("""
MATCH (s:Stations)
WHERE s.pk_m IS NULL
RETURN s.name AS station, s.PK AS raw_PK, s.line_code AS line_code, s.uic AS uic
ORDER BY station
"""))


,stations,with_pk_m
0,6062,6060


,station,raw_PK,line_code,uic
0,Musée-d'Orsay,D+936,984000,87547307
1,St-Michel-Notre-Dame,F+489,984000,87547315


In [112]:
# Create a Route node for Train 6161 and connect it to the main stops with an order
run_tx("""
MERGE (r:Route {id:'Train-6161-Paris-to-nice-direct'})

WITH r, [
  'Paris-Gare-de-Lyon',
  'Marseille-St-Charles',
  'Toulon',
  'Les Arcs-Draguignan',
  'Cannes',
  'Antibes',
  'Nice-Ville'
] AS stops

UNWIND range(0, size(stops)-1) AS i
MATCH (s:Stations {name: stops[i]})
MERGE (r)-[rel:SERVES]->(s)
SET rel.stop_order = i;
""")

# see the actual Paris -> NIce Route!
display(run_df("""
MATCH (r:Route {id:'Train-6161-Paris-to-nice-direct'})-[rel:SERVES]->(s:Stations)
RETURN rel.stop_order AS order, s.name AS station
ORDER BY order;
"""))

,order,station
0,0,Paris-Gare-de-Lyon
1,1,Marseille-St-Charles
2,2,Toulon
3,3,Les Arcs-Draguignan
4,4,Cannes
5,5,Antibes
6,6,Nice-Ville


In [113]:
# Create a Route node for Train 6173 and connect it to the main stops with an order
run_tx("""
MERGE (r:Route {id:'Train-6173-Paris-to-Marseille-St-Charles-direct'})

WITH r, [
  'Paris-Gare-de-Lyon',
  'Marseille-St-Charles'
] AS stops

UNWIND range(0, size(stops)-1) AS i
MATCH (s:Stations {name: stops[i]})
MERGE (r)-[rel:SERVES]->(s)
SET rel.stop_order = i;
""")

# see the actual Paris -> Marseille-St-Charles Route!
display(run_df("""
MATCH (r:Route {id:'Train-6173-Paris-to-Marseille-St-Charles-direct'})-[rel:SERVES]->(s:Stations)
RETURN rel.stop_order AS order, s.name AS station
ORDER BY order;
"""))

,order,station
0,0,Paris-Gare-de-Lyon
1,1,Marseille-St-Charles


In [114]:
#Route from Paris to NimesPondDuGard Train 6275
#This is the first part of the route it will have a connection
run_tx("""
MERGE (r:Route {id:'Train-6275-Paris-to-NimesPontduGard'})

WITH r, [
  'Paris-Gare-de-Lyon',
  'Valence',
  'Nîmes-Pont du Gard'
] AS stops

UNWIND range(0, size(stops)-1) AS i
MATCH (s:Stations {name: stops[i]})
MERGE (r)-[rel:SERVES]->(s)
SET rel.stop_order = i;
""")


In [115]:
#Route from Nimes PontDuGard to Port-Vendres Train 86987
#Connection from Paris
run_tx("""
MERGE (r:Route {id:'Train-86987-Nimes-to-PortVendres'})

WITH r, ['Nîmes',
        'Vergèze-Codognan', 
        'Lunel', 
        'Baillargues', 
        'Montpellier-St-Roch', 
        'Frontignan', 
        'Sète', 
        'Agde', 
        'Béziers', 
        'Narbonne', 
        'Port-la-Nouvelle', 
        'Leucate-la-Franqui', 
        'Salses', 
        'Rivesaltes', 
        'Perpignan', 
        'Elne', 
        'Argelès-sur-Mer', 
        'Collioure', 
        'Montpellier Sud-de-France', 
        'Port-Vendres'
        ] AS stops
UNWIND range(0, size(stops)-1) AS i
MATCH (s:Stations {name: stops[i]})
MERGE (r)-[rel:SERVES]->(s)
SET rel.stop_order = i;
""")

In [116]:
#Create NEXT_STOP edges along the Train-6275-Paris-to-NimesPontduGard
run_tx("""
MATCH (r:Route {id:'Train-6275-Paris-to-NimesPontduGard'})-[rel:SERVES]->(s:Stations)
WITH r, rel, s
ORDER BY rel.stop_order
WITH r, collect(s) AS stops

UNWIND range(0, size(stops)-2) AS i
WITH r, stops[i] AS a, stops[i+1] AS b, i
MERGE (a)-[rseg:NEXT_STOP {
  route_id: r.id,
  order_from: i
}]->(b);
""")

# sanity check
display(run_df("""
MATCH (a:Stations)-[r:NEXT_STOP {route_id:'Train-6275-Paris-to-NimesPontduGard'}]->(b:Stations)
RETURN r.order_from AS order_from, a.name AS from_station, b.name AS to_station
ORDER BY order_from;
"""))


,order_from,from_station,to_station
0,0,Paris-Gare-de-Lyon,Valence
1,1,Valence,Nîmes-Pont du Gard


In [117]:
#Create NEXT_STOP edges along the Train-86987-Nimes-to-PortVendres
run_tx("""
MATCH (r:Route {id:'Train-86987-Nimes-to-PortVendres'})-[rel:SERVES]->(s:Stations)
WITH r, rel, s
ORDER BY rel.stop_order
WITH r, collect(s) AS stops

UNWIND range(0, size(stops)-2) AS i
WITH r, stops[i] AS a, stops[i+1] AS b, i
MERGE (a)-[rseg:NEXT_STOP {
  route_id: r.id,
  order_from: i
}]->(b);
""")

# sanity check
display(run_df("""
MATCH (a:Stations)-[r:NEXT_STOP {route_id:'Train-86987-Nimes-to-PortVendres'}]->(b:Stations)
RETURN r.order_from AS order_from, a.name AS from_station, b.name AS to_station
ORDER BY order_from;
"""))

,order_from,from_station,to_station
0,0,Nîmes,Vergèze-Codognan
1,1,Vergèze-Codognan,Lunel
2,2,Lunel,Baillargues
3,3,Baillargues,Montpellier-St-Roch
4,4,Montpellier-St-Roch,Frontignan
5,5,Frontignan,Sète
6,6,Sète,Agde
7,7,Agde,Béziers
8,8,Béziers,Narbonne
9,9,Narbonne,Port-la-Nouvelle


In [118]:
# Create NEXT_STOP edges along the Train-6161 route in order
run_tx("""
MATCH (r:Route {id:'Train-6161-Paris-to-nice-direct'})-[rel:SERVES]->(s:Stations)
WITH r, rel, s
ORDER BY rel.stop_order
WITH collect(s) AS stops

UNWIND range(0, size(stops)-2) AS i
WITH stops[i] AS a, stops[i+1] AS b, i
MERGE (a)-[rseg:NEXT_STOP {
  route_id: 'Train-6161-Paris-to-nice-direct',
  order_from: i
}]->(b);
""")

# sanity check: show the segments
display(run_df("""
MATCH (a:Stations)-[r:NEXT_STOP {route_id:'Train-6161-Paris-to-nice-direct'}]->(b:Stations)
RETURN r.order_from AS order_from, a.name AS from_station, b.name AS to_station
ORDER BY order_from;
"""))



,order_from,from_station,to_station
0,0,Paris-Gare-de-Lyon,Marseille-St-Charles
1,1,Marseille-St-Charles,Toulon
2,2,Toulon,Les Arcs-Draguignan
3,3,Les Arcs-Draguignan,Cannes
4,4,Cannes,Antibes
5,5,Antibes,Nice-Ville


In [119]:
# Create NEXT_STOP edges along the Train-6173 route in order
run_tx("""
MATCH (r:Route {id:'Train-6173-Paris-to-Marseille-St-Charles-direct'})-[rel:SERVES]->(s:Stations)
WITH r, rel, s
ORDER BY rel.stop_order
WITH collect(s) AS stops

UNWIND range(0, size(stops)-2) AS i
WITH stops[i] AS a, stops[i+1] AS b, i
MERGE (a)-[rseg:NEXT_STOP {
  route_id: 'Train-6173-Paris-to-Marseille-St-Charles-direct',
  order_from: i
}]->(b);
""")

# sanity check: show the 6173 NEXT_STOP segments
display(run_df("""
MATCH (a:Stations)-[r:NEXT_STOP {route_id:'Train-6173-Paris-to-Marseille-St-Charles-direct'}]->(b:Stations)
RETURN r.order_from AS order_from, a.name AS from_station, b.name AS to_station
ORDER BY order_from;
"""))

,order_from,from_station,to_station
0,0,Paris-Gare-de-Lyon,Marseille-St-Charles


In [120]:
# route for Train 6161 
display(run_df("""
MATCH (r:Route {id:'Train-6161-Paris-to-nice-direct'})-[rel:SERVES]->(s:Stations)
WITH r, rel, s
ORDER BY rel.stop_order
WITH collect(s) AS stop_nodes
RETURN size(stop_nodes) - 1 AS hops,
       [n IN stop_nodes | n.name] AS stations;
"""))

,hops,stations
0,6,"[Paris-Gare-de-Lyon, Marseille-St-Charles, Toulon, Les Arcs-Draguignan, Cannes, Antibes, Nice-Ville]"


In [121]:
#Connection between 6275 and 86987
run_tx("""
MATCH (r1:Route {id:'Train-6275-Paris-to-NimesPontduGard'}),
      (r2:Route {id:'Train-86987-Nimes-to-PortVendres'})
MERGE (r1)-[c:CONNECTION {time:15}]->(r2);
""")

# check the connection
display(run_df("""
MATCH (r1:Route)-[c:CONNECTION]->(r2:Route)
RETURN r1.id AS from_route, r2.id AS to_route, c.time AS connection_minutes;
"""))


,from_route,to_route,connection_minutes
0,Train-6275-Paris-to-NimesPontduGard,Train-86987-Nimes-to-PortVendres,15


In [122]:
import neo4j

import pandas as pd

from IPython.display import display
driver1 = neo4j.GraphDatabase.driver(uri="neo4j://localhost:7687", auth=("neo4j","<your password>"))
session = driver1.session(database="neo4j")

In [123]:
def my_neo4j_run_query_pandas(query, **kwargs):
    "run a query and return the results in a pandas dataframe"
    
    result = session.run(query, **kwargs)
    
    df = pd.DataFrame([r.values() for r in result], columns=result.keys())
    
    return df

In [124]:
def my_neo4j_nodes_relationships():
    "print all the nodes and relationships"
   
    print("-------------------------")
    print("  Nodes:")
    print("-------------------------")
    
    query = """
        match (n) 
        return n.name as node_name, labels(n) as labels
        order by n.name
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_nodes = df.shape[0]
    
    display(df)
    
    print("-------------------------")
    print("  Relationships:")
    print("-------------------------")
    
    query = """
        match (n1)-[r]->(n2) 
        return n1.name as node_name_1, labels(n1) as node_1_labels, 
            type(r) as relationship_type, n2.name as node_name_2, labels(n2) as node_2_labels
        order by node_name_1, node_name_2
    """
    
    df = my_neo4j_run_query_pandas(query)
    
    number_relationships = df.shape[0]
    
    display(df)
    
    density = (2 * number_relationships) / (number_nodes * (number_nodes - 1))
    
    print("-------------------------")
    print("  Density:", f'{density:.1f}')
    print("-------------------------")

In [125]:
my_neo4j_nodes_relationships()

-------------------------
  Nodes:
-------------------------


,node_name,labels
0,Abancourt,[Stations]
1,Abbaretz,[Stations]
2,Abbecourt,[Stations]
3,Abbeville,[Stations]
4,Abilly,[Stations]
...,...,...
7054,None,[Line]
7055,None,[Route]
7056,None,[Route]
7057,None,[Route]


-------------------------
  Relationships:
-------------------------


,node_name_1,node_1_labels,relationship_type,node_name_2,node_2_labels
0,Abancourt,[Stations],TRACK,Formerie,[Stations]
1,Abancourt,[Stations],TRACK,Romescamps,[Stations]
2,Abancourt,[Stations],ON_LINE,None,[Line]
3,Abbaretz,[Stations],TRACK,Issé,[Stations]
4,Abbaretz,[Stations],TRACK,Saffré,[Stations]
...,...,...,...,...,...
20417,None,[Route],SERVES,Sète,[Stations]
20418,None,[Route],SERVES,Toulon,[Stations]
20419,None,[Route],SERVES,Valence,[Stations]
20420,None,[Route],SERVES,Vergèze-Codognan,[Stations]


-------------------------
  Density: 0.0
-------------------------


In [126]:

session.run("CALL gds.graph.drop('ds_graph', false) YIELD graphName")

# Project Stations + TRACK into a named graph for GDS
session.run("""
CALL gds.graph.project(
  'ds_graph',
  'Stations',                 
  {
    TRACK: {
      type: 'TRACK',
      properties: 'weight_m'   
    }
  }
)
""")


## Centrality

In [127]:
df_isolated = my_neo4j_run_query_pandas("""
MATCH (s:Stations)
WHERE NOT EXISTS { (s)-[:TRACK]-() }
RETURN s.name AS station
ORDER BY station
""")
display(df_isolated)


,station
0,Anizy-Pinon
1,Anor
2,Arudy
3,Audun-le-Tiche
4,Bagnoles-de-l'Orne-Tessé-la-Madeleine
...,...
61,Vierzy
62,Villers-Cotterêts
63,Vivier-au-Court
64,Voyenne


In [128]:
df_degree_dist = my_neo4j_run_query_pandas("""
MATCH (s:Stations)
RETURN
  COUNT { (s)-[:TRACK]-() } AS degree,
  COUNT(*) AS stations
ORDER BY degree
""")
display(df_degree_dist)


,degree,stations
0,0,66
1,2,659
2,4,4371
3,6,448
4,8,173
5,10,74
6,12,63
7,14,35
8,16,34
9,18,24


In [129]:
df_components = my_neo4j_run_query_pandas("""
CALL gds.wcc.stream('ds_graph')
YIELD nodeId, componentId
RETURN componentId, count(*) AS size
ORDER BY size DESC
""")
display(df_components)


,componentId,size
0,2,3210
1,25,121
2,1,74
3,50,72
4,80,69
...,...,...
308,5396,1
309,5414,1
310,5823,1
311,5835,1


In [130]:
df_giant = my_neo4j_run_query_pandas("""
CALL gds.wcc.stream('ds_graph')
YIELD nodeId, componentId
WITH componentId, collect(nodeId) AS nodes
ORDER BY size(nodes) DESC
LIMIT 1
UNWIND nodes AS nodeId
RETURN gds.util.asNode(nodeId).name AS station
ORDER BY station
""")
display(df_giant)


,station
0,Abancourt
1,Abbaretz
2,Abbecourt
3,Abbeville
4,Ablon
...,...
3205,Évires
3206,Évron
3207,Évry-Courcouronnes
3208,Île-Napoléon


In [131]:
# Degree Centrality
df_deg = my_neo4j_run_query_pandas("""
CALL gds.degree.stream('ds_graph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS station,
       score AS degree
ORDER BY degree DESC
LIMIT 20
""")
display(df_deg)


,station,degree
0,Paris-La Chapelle,18.0
1,Paris-Bercy-Charolais,16.0
2,Paris-Bercy-Conflans,16.0
3,Champ-de-Mars-Tour-Eiffel,16.0
4,Invalides,16.0
5,Issy-Val-de-Seine,16.0
6,Clichy-Levallois,16.0
7,Haussmann-St-Lazare,16.0
8,Paris-Bercy-Rapée,16.0
9,Porte Maillot,16.0


In [132]:
#Between Centrality
df_between = my_neo4j_run_query_pandas("""
CALL gds.betweenness.stream('ds_graph')
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS station,
       score AS betweenness
ORDER BY betweenness DESC
LIMIT 20
""")
display(df_between)


,station,betweenness
0,Valenton,4.854602e+06
1,Le Mée,4.050045e+06
2,St-Mammès,4.044282e+06
3,Bois-le-Roi,4.042825e+06
4,Fontainebleau-Avon,4.032694e+06
5,Thomery,4.013645e+06
6,Villeneuve-St-Georges,3.978917e+06
7,Montgeron-Crosne,3.975996e+06
8,Yerres,3.973075e+06
9,Brunoy,3.970154e+06


In [133]:
df_close = my_neo4j_run_query_pandas("""
CALL gds.closeness.stream('ds_graph', {
  relationshipTypes: ['TRACK'],
  useWassermanFaust: true  
})
YIELD nodeId, score
RETURN gds.util.asNode(nodeId).name AS station,
       score AS closeness
ORDER BY closeness DESC
LIMIT 20
""")
display(df_close)


,station,closeness
0,Valenton,0.007544
1,Sucy-Bonneuil,0.007528
2,La Varenne-Chennevières,0.007525
3,Villeneuve-Triage,0.007520
4,Bobigny,0.007505
5,Petit-Vaux,0.007490
6,Villeneuve-Prairie,0.007483
7,Créteil-Pompadour,0.007483
8,Villeneuve-St-Georges,0.007474
9,La Courneuve-Dugny,0.007467


## Shortest paths

In [134]:
# associating shorteest paths via lat/long
run_tx("""
MATCH (a:Stations)-[r:TRACK]->(b:Stations)
WHERE a.location IS NOT NULL AND b.location IS NOT NULL
  AND (r.weight_m IS NULL OR r.weight_m = 0)
SET r.weight_m = point.distance(a.location, b.location)
""")

# sanity
display(run_df("""
MATCH (a:Stations)-[r:TRACK]->(b:Stations)
RETURN a.name AS from_station, b.name AS to_station, r.weight_m AS weight_m
LIMIT 10
"""))


,from_station,to_station,weight_m
0,Cuzorn,St-Front-sur-Lémance,5269.00000
1,Cuzorn,Trentels-Ladignac,12715.00000
2,Gamarde-les-Bains,Hinx-sur-Adour,4322.00000
3,Gamarde-les-Bains,Montfort-en-Chalosse,3220.00000
4,Vars,St-Amant-de-Boixe,5186.00000
5,Vars,Angoulême,13556.00000
6,Monte-Carlo-Country-Club,Carnolès,2727.13452
7,Monte-Carlo-Country-Club,Cap-d'Ail,6552.00000
8,Monte-Carlo-Country-Club,Cap-Martin-Roquebrune,1007.00000
9,La Boussac,Plerguer,17184.00000


In [135]:
df_drop_ds_graph = my_neo4j_run_query_pandas("""
CALL gds.graph.drop('ds_graph', false)
YIELD graphName
RETURN graphName
""")
display(df_drop_ds_graph)

df_project_ds_graph = my_neo4j_run_query_pandas("""
CALL gds.graph.project(
  'ds_graph',
  'Stations',
  { TRACK: { orientation: 'UNDIRECTED' } },
  { relationshipProperties: ['weight_m'] }
)
YIELD graphName, nodeCount, relationshipCount
""")
display(df_project_ds_graph)



,graphName
0,ds_graph


,graphName,nodeCount,relationshipCount
0,ds_graph,6062,28728


In [144]:
df_paris_nice = my_neo4j_run_query_pandas("""
MATCH (source:Stations {name:'Paris-Gare-de-Lyon'}), (target:Stations {name:'Nice-Ville'})
CALL gds.shortestPath.dijkstra.stream('ds_graph', {
  sourceNode: id(source),
  targetNode: id(target),
  relationshipWeightProperty: 'weight_m'
})
YIELD totalCost, nodeIds
RETURN [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS stations, totalCost AS total_distance_m
""")
display(df_paris_nice)

Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description='The query used a deprecated function: `id`.', position=<SummaryInputPosition line=4, column=15, offset=158>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'column': 15, 'offset': 158, 'line': 4}}> for query: "\nMATCH (source:Stations {name:'Paris-Gare-de-Lyon'}), (target:Stations {name:'Nice-Ville'})\nCALL gds.shortestPath.dijkstra.stream('ds_graph', {\n  sourceNode: id(source),\n  targetNode: id(target),\n  relationshipWeightProperty: 'weight_m'\n})\nYIELD totalCost, nodeIds\nRETURN [nodeId IN nodeIds | gds.util.asNode(nodeId).name] AS stations, totalCost AS total_distance_m\n"
Received notification from 

,stations,total_distance_m
0,"[Paris-Gare-de-Lyon, Paris-Bercy-Rapée, Ivry-sur-Seine, Vitry-sur-Seine, Les Ardoines, Villeneuve-Prairie, Villeneuve-Triage, Villeneuve-St-Georges, Montgeron-Crosne, Yerres, Brunoy, Boussy-St-Antoine, Combs-la-Ville-Quincy, Lieusaint-Moissy, Savigny-le-Temple-Nandy, Cesson, Le Mée, Bois-le-Roi, Fontainebleau-Avon, Thomery, St-Mammès, Montereau, Villeneuve-la-Guyard, Champigny-sur-Yonne, Pont-sur-Yonne, Sens, Étigny-Véron, Villeneuve-sur-Yonne, St-Julien-du-Sault, Cézy, Joigny, Laroche-Migennes, Brienon, Flogny, Tonnerre, Tanlay, Lézinnes, Port-Sec-de-Pacy, Ancy-le-Franc, Aisy, Montbard, Fain-lès-Montbard, Les Laumes-Alésia, Darcey (Côte-d'Or), Thenissey, Verrey, Blaisy-Bas, Mâlain, Lantenay, Velars, Plombières, Dijon-Ville, Dijon-Porte-d'Ouche-Perrigny, Gevrey-Chambertin, Vougeot-Gilly-lès-Cîteaux, Nuits-St-Georges, Corgoloin, Serrigny, Beaune, Meursault, Chagny, Rully, Fontaines-Mercurey, Chalon-sur-Saône, Varennes-le-Grand, Sennecey-le-Grand, Tournus, Uchizy, Fleurville-Pont-de-Vaux, Senozan, Mâcon-Ville, Crêches-sur-Saône, Pontanevaux, Romanèche-Thorins, Belleville-sur-Saône, St-Georges-de-Reneins, Villefranche-sur-Saône, Anse, Quincieux-Trévoux, Quincieux, St-Germain-au-Mont-d'Or, Albigny-Neuville, Couzon-au-Mont-d'Or, Fontaines-sur-Saône, Collonges-Fontaines, Lyon-Vaise, Lyon-Gorge-de-Loup, Lyon-Perrache (MIN), Oullins, Pierre-Bénite, Irigny-Yvours, Vernaison, Grigny, Le Sablon, Chasse-sur-Rhône, Loire, Ste-Colombe-les-Vienne-St-Romain-en-Gal, Ampuis, Condrieu, St-Clair-les-Roches, ...]",1.079965e+06


In [137]:
df_fence_hops = my_neo4j_run_query_pandas("""
MATCH (start:Stations {name:'Paris-Gare-de-Lyon'})
MATCH path = (start)-[:TRACK*1..3]-(s:Stations)
WITH start, s, min(length(path)) AS hops
WHERE s <> start
RETURN s.name AS station, hops
ORDER BY hops ASC, station ASC
""")
display(df_fence_hops)

,station,hops
0,Bibliothèque-François-Mitterrand,1
1,Boulevard-Masséna,1
2,Centre Henri Lang,1
3,Charonne-Ceinture,1
4,Chatelet-les-Halles,1
5,Paris-Austerlitz,1
6,Paris-Austerlitz-Souterrain,1
7,Paris-Bercy,1
8,Paris-Bercy-Charolais,1
9,Paris-Bercy-Conflans,1


In [138]:
df_fence_distance = my_neo4j_run_query_pandas("""
MATCH (start:Stations {name:'Paris-Gare-de-Lyon'})
CALL gds.allShortestPaths.dijkstra.stream('ds_graph', {
  sourceNode: id(start),
  relationshipWeightProperty: 'weight_m'
})
YIELD targetNode, totalCost
WITH start, gds.util.asNode(targetNode) AS target, totalCost
WHERE target <> start AND totalCost <= 300000
RETURN target.name AS station, totalCost AS distance_m
ORDER BY distance_m ASC, station ASC
""")
display(df_fence_distance)

Received notification from DBMS server: <GqlStatusObject gql_status='01N42', status_description='The query used a deprecated function: `id`.', position=<SummaryInputPosition line=4, column=15, offset=122>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/', '_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'column': 15, 'offset': 122, 'line': 4}}> for query: "\nMATCH (start:Stations {name:'Paris-Gare-de-Lyon'})\nCALL gds.allShortestPaths.dijkstra.stream('ds_graph', {\n  sourceNode: id(start),\n  relationshipWeightProperty: 'weight_m'\n})\nYIELD targetNode, totalCost\nWITH start, gds.util.asNode(targetNode) AS target, totalCost\nWHERE target <> start AND totalCost <= 300000\nRETURN target.name AS station, totalCost AS distance_m\nORDER BY distance_m ASC, st

,station,distance_m
0,Paris-Gare-de-Lyon-Souterrain,0.000000
1,Centre Henri Lang,180.000000
2,Paris-Austerlitz,760.371331
3,Paris-Austerlitz-Souterrain,760.371331
4,Paris-Bercy,834.352361
...,...,...
1491,Verquigneul,298426.696246
1492,Vis-à-Marles,298549.363022
1493,Lantenay,299089.936295
1494,Verneuil (Nièvre),299448.245087
